[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap03/cap03.EPs_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

## 💻 **Parte Prática com Exercícios de Programação**

---

### 🎯 Objetivo deste Caderno

O caderno permite desenvolver, validar, organizar e testar soluções de **Exercícios de Programação (EPs)** em ambientes interativos, como o Colab, com os mesmos casos de teste do Moodle, copiando para lá apenas na hora de registrar a nota oficial.

In [1]:
import os, sys, importlib, inspect, urllib.request

BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py", "testsuite.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph, testsuite
importlib.reload(morph); importlib.reload(testsuite)
from morph import mm
from testsuite import TestSuite

print(f"✅ Ambiente pronto. Morph: {morph.__version__} | TestSuite: {testsuite.__version__}")

✅ Ambiente pronto. Morph: 1.1.0 | TestSuite: 1.1.0


---

#### Executando os Testes

Para rodar os testes, execute `TestSuite("EP03_XX.extensão").run()` numa nova célula, trocando a extensão pela da linguagem usada (`.py`, `.java`, `.c`, `.cpp`, `.js` ou `.r`). O sistema baixa os casos de teste do GitHub, executa o programa e calcula a nota automaticamente.

### EP03_01 ☀️ Transformação Linear de Intensidade

Neste exercício você implementará um operador de ponto para **ajuste dinâmico de brilho e contraste**, baseado na transformação $p' = \text{clip}(\text{round}(\alpha \cdot p + \beta))$. Esta operação é fundamental em pipelines de pré-processamento para normalizar variações de iluminação.

#### 📋 Diretrizes de Implementação

1. Leia as dimensões $L$ (linhas) e $C$ (colunas).
2. Leia os parâmetros $\alpha$ (real, fator de contraste) e $\beta$ (inteiro, fator de brilho).
3. Leia a matriz original (valores inteiros entre 0 e 255).
4. Para cada pixel $p$, calcule $p' = \text{round}(\alpha \cdot p + \beta)$ e depois aplique saturação: $p' = \max(0, \min(255, p'))$.
5. Exiba a matriz resultante com $L$ linhas e $C$ colunas.

---

#### 📌 Restrições e Fórmulas

$$p' = \text{clip}(\text{round}(\alpha \cdot p + \beta)), \quad \text{clip}(x) = \max(0, \min(255, x))$$

| Parâmetro | Efeito |
|-----------|--------|
| $\alpha > 1$ | Aumenta o contraste (expande o histograma) |
| $0 \le \alpha < 1$ | Reduz o contraste (comprime o histograma) |
| $\beta > 0$ | Aumenta o brilho (desloca histograma para direita) |
| $\beta < 0$ | Diminui o brilho (desloca para esquerda) |

---

#### 📦 Entrada e Saída (VPL)

**Entrada:**
- Linha 1: $L$
- Linha 2: $C$
- Linha 3: $\alpha$ $\beta$
- $L$ linhas seguintes: $C$ inteiros cada (matriz original)

**Saída:**
- Matriz transformada, $L$ linhas com $C$ inteiros cada.

| Exemplo de Entrada | Saída Esperada |
|--------------------|----------------|
| 1<br>4<br>1.5 -30<br>0 100 180 255 | 0 120 240 255 |


In [2]:
#| label: fig-ep03_01-sim
#| fig-cap: "Simulador: Transformação Linear de Intensidade (Brilho e Contraste)"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_01_sim" style="background-color: #fef9ef; border-radius: 18px; border: 1px solid #ede6d8; overflow: hidden; margin-top: 20px; font-family: sans-serif;">
    <div style="background: #f3efe6; padding: 8px 16px; font-size: 12px; color: #5e5a4a; border-bottom: 1px solid #e9dfcf; display: flex; justify-content: space-between;">
        <span>🎮 Simulador: Brilho e Contraste</span>
        <span style="background: #e8e0cf; border-radius: 40px; padding: 2px 10px; font-weight: 600; font-size: 10px;">p' = clip(round(α·p + β))</span>
    </div>
    <div style="padding: 20px; background: white;">
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 30px; background: #fafafa; border-radius: 12px; padding: 20px; margin-bottom: 20px;">
            <div><label>α (Contraste)</label><input type="range" id="ep03_01_alpha" min="0" max="3" step="0.05" value="1.0"><span id="ep03_01_alpha_val" style="margin-left: 10px;">1.00</span></div>
            <div><label>β (Brilho)</label><input type="range" id="ep03_01_beta" min="-100" max="100" step="1" value="0"><span id="ep03_01_beta_val" style="margin-left: 10px;">0</span></div>
        </div>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="text-align: center; border: 1px solid #eee; border-radius: 12px; padding: 15px;">
                <p style="font-size: 10px; font-weight: bold;">Original</p>
                <div id="ep03_01_grid_orig" style="display: grid; grid-template-columns: repeat(4, 42px); gap: 4px; justify-content: center;"></div>
                <button id="ep03_01_random" style="margin-top: 10px;">🎲 Nova Imagem</button>
            </div>
            <div style="text-align: center; border: 1px solid #eee; border-radius: 12px; padding: 15px;">
                <p style="font-size: 10px; font-weight: bold;">Resultado</p>
                <div id="ep03_01_grid_new" style="display: grid; grid-template-columns: repeat(4, 42px); gap: 4px; justify-content: center;"></div>
                <button id="ep03_01_reset" style="margin-top: 10px;">↩ Reset</button>
            </div>
        </div>
        <div id="ep03_01_debug" style="margin-top: 20px; background: #e8f5e9; border-radius: 8px; padding: 10px; font-family: monospace; font-size: 12px; text-align: center;"></div>
    </div>
</div>
<script>
setTimeout(() => {
    const alphaSlider = document.getElementById('ep03_01_alpha');
    const betaSlider = document.getElementById('ep03_01_beta');
    const alphaSpan = document.getElementById('ep03_01_alpha_val');
    const betaSpan = document.getElementById('ep03_01_beta_val');
    const gridOrig = document.getElementById('ep03_01_grid_orig');
    const gridNew = document.getElementById('ep03_01_grid_new');
    const debugDiv = document.getElementById('ep03_01_debug');
    let pixels = Array(16).fill(0).map(() => Math.floor(Math.random() * 256));
    function render() {
        const a = parseFloat(alphaSlider.value);
        const b = parseInt(betaSlider.value);
        alphaSpan.innerText = a.toFixed(2);
        betaSpan.innerText = b;
        debugDiv.innerHTML = `Fórmula: clip(round(${a.toFixed(2)} * p + ${b}))`;
        gridOrig.innerHTML = '';
        gridNew.innerHTML = '';
        pixels.forEach(p => {
            const origDiv = document.createElement('div');
            origDiv.style.cssText = 'width:42px; height:42px; display:flex; align-items:center; justify-content:center; font-weight:bold; border-radius:4px; border:1px solid #ddd;';
            origDiv.style.backgroundColor = `rgb(${p},${p},${p})`;
            origDiv.style.color = p > 128 ? 'black' : 'white';
            origDiv.innerText = p;
            gridOrig.appendChild(origDiv);
            let out = Math.round(a * p + b);
            out = Math.max(0, Math.min(255, out));
            const newDiv = document.createElement('div');
            newDiv.style.cssText = 'width:42px; height:42px; display:flex; align-items:center; justify-content:center; font-weight:bold; border-radius:4px; border:1px solid #ddd;';
            newDiv.style.backgroundColor = `rgb(${out},${out},${out})`;
            newDiv.style.color = out > 128 ? 'black' : 'white';
            newDiv.innerText = out;
            gridNew.appendChild(newDiv);
        });
    }
    alphaSlider.oninput = render;
    betaSlider.oninput = render;
    document.getElementById('ep03_01_random').onclick = () => { pixels = Array(16).fill(0).map(() => Math.floor(Math.random() * 256)); render(); };
    document.getElementById('ep03_01_reset').onclick = () => { alphaSlider.value = 1; betaSlider.value = 0; render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_01.py
# Código Python

In [ ]:
TestSuite("EP03_01.py").run()

### EP03_02 ➕ Operações Aritméticas Saturadas (Adição e Subtração)

A adição e subtração de imagens são usadas para fusão, remoção de fundo e realce de diferenças. Implemente as funções `add_sat` e `sub_sat` que operam pixel a pixel com saturação no intervalo [0,255].

#### 📋 Especificação

1. Leia $L$, $C$ e duas matrizes $A$ e $B$ (mesmas dimensões).
2. Calcule $C = A + B$ com saturação: $c = \max(0, \min(255, a+b))$.
3. Calcule $D = A - B$ com saturação: $d = \max(0, \min(255, a-b))$.
4. Imprima $C$ e depois $D$, cada matriz com $L$ linhas e $C$ colunas.

---

#### 🧮 Fórmulas

$$\text{add}(a,b) = \text{clip}(a+b), \quad \text{sub}(a,b) = \text{clip}(a-b)$$

#### 📊 Exemplo

| Entrada | Saída (adição) | Saída (subtração) |
|---------|----------------|-------------------|
| 1 3<br>200 50 30<br>100 200 30 | 255 250 60 | 100 0 0 |


In [3]:
#| label: fig-ep03_02-sim
#| fig-cap: "Simulador: Adição e Subtração Saturada"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_02_sim" style="background-color: #fef9ef; border-radius: 18px; border: 1px solid #ede6d8; margin-top: 20px; font-family: sans-serif;">
    <div style="background: #f3efe6; padding: 8px 16px; display: flex; justify-content: space-between;">
        <span>➕ Simulador: Adição e Subtração Saturada</span>
    </div>
    <div style="padding: 20px; background: white;">
        <div style="display: grid; grid-template-columns: repeat(3,1fr); gap: 20px;">
            <div><p>Matriz A</p><div id="ep03_02_gridA"></div><button id="ep03_02_randA">Random A</button></div>
            <div><p>Matriz B</p><div id="ep03_02_gridB"></div><button id="ep03_02_randB">Random B</button></div>
            <div><p>A + B (clip)</p><div id="ep03_02_gridAdd"></div></div>
            <div><p>A - B (clip)</p><div id="ep03_02_gridSub"></div></div>
        </div>
    </div>
</div>
<script>
setTimeout(() => {
    function randomMat() { return Array(16).fill(0).map(() => Math.floor(Math.random() * 256)); }
    let matA = randomMat(), matB = randomMat();
    const gridA = document.getElementById('ep03_02_gridA');
    const gridB = document.getElementById('ep03_02_gridB');
    const gridAdd = document.getElementById('ep03_02_gridAdd');
    const gridSub = document.getElementById('ep03_02_gridSub');
    function render() {
        const renderMat = (mat, container) => {
            container.innerHTML = '';
            mat.forEach(v => {
                const d = document.createElement('div');
                d.style.cssText = 'width:42px; height:42px; display:inline-flex; margin:2px; align-items:center; justify-content:center; background:rgb('+v+','+v+','+v+'); color:'+(v>128?'black':'white')+'; border:1px solid #ccc;';
                d.innerText = v;
                container.appendChild(d);
            });
            container.style.display = 'grid';
            container.style.gridTemplateColumns = 'repeat(4,42px)';
        };
        const addSat = (a,b) => Math.min(255, Math.max(0, a+b));
        const subSat = (a,b) => Math.min(255, Math.max(0, a-b));
        const matAdd = matA.map((v,i) => addSat(v, matB[i]));
        const matSub = matA.map((v,i) => subSat(v, matB[i]));
        renderMat(matA, gridA); renderMat(matB, gridB);
        renderMat(matAdd, gridAdd); renderMat(matSub, gridSub);
    }
    document.getElementById('ep03_02_randA').onclick = () => { matA = randomMat(); render(); };
    document.getElementById('ep03_02_randB').onclick = () => { matB = randomMat(); render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_02.py
# Código Python

In [ ]:
TestSuite("EP03_02.py").run()

### EP03_03 🎭 Máscaras Bit a Bit (AND, OR, NOT)

Implemente operações lógicas bit a bit para isolar regiões de interesse (ROI). A máscara binária (0 ou 255) será fornecida juntamente com a imagem.

#### 📋 Tarefas

1. Ler $L$, $C$, a imagem $I$ (uint8) e a máscara $M$ (0/255).
2. Calcular: $I_{\text{AND}} = I \& M$, $I_{\text{OR}} = I | M$, $I_{\text{NOT}} = \text{NOT}(I)$ (bitwise).
3. Imprimir as três matrizes resultantes na ordem: AND, OR, NOT.

---

#### 📐 Representação Matemática

$$\text{AND}(i,m) = i \& m,\quad \text{OR}(i,m) = i | m,\quad \text{NOT}(i) = 255 - i$$

#### 📊 Exemplo (matriz 1x4)

Entrada: `1 4\n100 150 200 50\n0 255 0 255`  
Saída AND: 0 150 0 50  
Saída OR: 100 255 200 255  
Saída NOT: 155 105 55 205


In [4]:
#| label: fig-ep03_03-sim
#| fig-cap: "Simulador: Operações Lógicas com Máscara"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_03_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div style="display: grid; grid-template-columns: repeat(4,1fr); gap: 10px;">
        <div><strong>Imagem</strong><div id="ep03_03_img"></div><button id="ep03_03_randImg">Random</button></div>
        <div><strong>Máscara</strong><div id="ep03_03_mask"></div><button id="ep03_03_randMask">Random Mask</button></div>
        <div><strong>AND</strong><div id="ep03_03_and"></div></div>
        <div><strong>OR</strong><div id="ep03_03_or"></div></div>
        <div><strong>NOT (da imagem)</strong><div id="ep03_03_not"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let img = Array(16).fill(0).map(()=>Math.floor(Math.random()*256));
    let mask = Array(16).fill(0).map(()=>Math.random()<0.5?0:255);
    function renderGrid(container, arr) {
        container.innerHTML = '';
        arr.forEach(v=>{
            let d=document.createElement('div');
            d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid gray;';
            d.innerText=v;
            container.appendChild(d);
        });
        container.style.display='grid'; container.style.gridTemplateColumns='repeat(4,42px)';
    }
    function refresh() {
        renderGrid(document.getElementById('ep03_03_img'), img);
        renderGrid(document.getElementById('ep03_03_mask'), mask);
        let andRes = img.map((v,i)=> v & mask[i]);
        let orRes = img.map((v,i)=> v | mask[i]);
        let notRes = img.map(v=> 255 - v);
        renderGrid(document.getElementById('ep03_03_and'), andRes);
        renderGrid(document.getElementById('ep03_03_or'), orRes);
        renderGrid(document.getElementById('ep03_03_not'), notRes);
    }
    document.getElementById('ep03_03_randImg').onclick = ()=>{ img = Array(16).fill(0).map(()=>Math.floor(Math.random()*256)); refresh(); };
    document.getElementById('ep03_03_randMask').onclick = ()=>{ mask = Array(16).fill(0).map(()=>Math.random()<0.5?0:255); refresh(); };
    refresh();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_03.py
# Código Python

In [ ]:
TestSuite("EP03_03.py").run()

### EP03_04 📊 Equalização de Histograma (CDF)

Implemente o algoritmo clássico de equalização global: $s_k = (L-1) \sum_{j=0}^{k} p(r_j)$. Aplique a LUT resultante a cada pixel.

#### 📝 Passos

1. Calcular histograma $h[k]$ para $k=0..255$.
2. Calcular $p[k] = h[k] / (L \cdot C)$.
3. Calcular CDF: $cdf[k] = \sum_{j=0}^{k} p[j]$.
4. Mapeamento: $lut[k] = \text{round}(cdf[k] \cdot 255)$.
5. Aplicar $lut$ à imagem original.

#### 📥 Entrada / 📤 Saída

Entrada: $L$, $C$, matriz original. Saída: matriz equalizada.

| Entrada (3x3) | Saída (exemplo) |
|---------------|----------------|
| 50 100 150<br>50 100 150<br>50 100 150 | 0 127 255<br>0 127 255<br>0 127 255 |


In [5]:
#| label: fig-ep03_04-sim
#| fig-cap: "Simulador: Equalização de Histograma"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_04_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_04_rand">Gerar Imagem 4x4 Aleatória</button></div>
    <div style="display: grid; grid-template-columns: 1fr 1fr;">
        <div><strong>Original</strong><div id="ep03_04_orig"></div></div>
        <div><strong>Equalizada (mm.equalize)</strong><div id="ep03_04_eq"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    function genRand() { return Array(16).fill(0).map(()=>Math.floor(Math.random()*256)); }
    let img = genRand();
    function equalize(arr) {
        let hist = Array(256).fill(0);
        arr.forEach(v=>hist[v]++);
        let cdf = 0;
        let lut = Array(256);
        for(let i=0;i<256;i++) { cdf += hist[i]/16; lut[i] = Math.round(cdf*255); }
        return arr.map(v=>lut[v]);
    }
    function render() {
        let eq = equalize(img);
        const renderMat = (container, mat) => {
            container.innerHTML = '';
            mat.forEach(v=>{
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            });
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(4,42px)';
        };
        renderMat(document.getElementById('ep03_04_orig'), img);
        renderMat(document.getElementById('ep03_04_eq'), eq);
    }
    document.getElementById('ep03_04_rand').onclick = () => { img = genRand(); render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_04.py
# Código Python

In [ ]:
TestSuite("EP03_04.py").run()

### EP03_05 🎨 Especificação de Histograma (Transferência de Perfil)

Implemente a especificação de histograma: mapeie a imagem de entrada para que seu perfil tonal se aproxime de uma imagem de referência.

#### 🔧 Método

1. Calcule CDF da entrada $F_e$ e CDF da referência $F_r$.
2. Para cada nível $r$, encontre $z$ que minimiza $|F_r(z) - F_e(r)|$.
3. Construa LUT e aplique.

#### 📥 Entrada

$L$, $C$, matriz origem, matriz referência (mesmas dimensões). Saída: matriz especificada.

| Exemplo (1x4) |
|---------------|
| Origem: 0 100 200 255  <br> Referência: 0 0 255 255  <br> Saída esperada: 0 0 255 255 |


In [6]:
#| label: fig-ep03_05-sim
#| fig-cap: "Simulador: Especificação de Histograma"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_05_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_05_rand">Gerar Origem e Referência</button></div>
    <div style="display: grid; grid-template-columns: repeat(3,1fr);">
        <div><strong>Original</strong><div id="ep03_05_orig"></div></div>
        <div><strong>Referência</strong><div id="ep03_05_ref"></div></div>
        <div><strong>Especificada</strong><div id="ep03_05_spec"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let orig = Array(16).fill(0).map(()=>Math.floor(Math.random()*256));
    let ref = Array(16).fill(0).map(()=>Math.floor(Math.random()*256));
    function cdf(arr) {
        let h = Array(256).fill(0); arr.forEach(v=>h[v]++);
        let cum=0, cdfArr=Array(256);
        for(let i=0;i<256;i++) { cum += h[i]/arr.length; cdfArr[i]=cum; }
        return cdfArr;
    }
    function specify(src, target) {
        let cdf_src = cdf(src);
        let cdf_tgt = cdf(target);
        let lut = Array(256);
        for(let i=0;i<256;i++) {
            let best = 0, bestDiff = Math.abs(cdf_tgt[0] - cdf_src[i]);
            for(let j=1;j<256;j++) {
                let diff = Math.abs(cdf_tgt[j] - cdf_src[i]);
                if(diff < bestDiff) { bestDiff=diff; best=j; }
            }
            lut[i]=best;
        }
        return src.map(v=>lut[v]);
    }
    function render() {
        let spec = specify(orig, ref);
        function draw(container, arr) {
            container.innerHTML='';
            arr.forEach(v=>{
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            });
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(4,42px)';
        }
        draw(document.getElementById('ep03_05_orig'), orig);
        draw(document.getElementById('ep03_05_ref'), ref);
        draw(document.getElementById('ep03_05_spec'), spec);
    }
    document.getElementById('ep03_05_rand').onclick = () => { orig = Array(16).fill(0).map(()=>Math.floor(Math.random()*256)); ref = Array(16).fill(0).map(()=>Math.floor(Math.random()*256)); render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_05.py
# Código Python

In [ ]:
TestSuite("EP03_05.py").run()

### EP03_06 🔄 Correlação vs Convolução com Kernel Sobel

Implemente manualmente a correlação cruzada e a convolução (rotacionando o kernel 180°) e compare os resultados usando o kernel Sobel $G_x$.

#### 📐 Definições

$$ \text{correlação}: g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t) f(x+s,y+t) $$
$$ \text{convolução}: g(x,y) = \sum_{s=-a}^{a}\sum_{t=-b}^{b} w(s,t) f(x-s,y-t) $$

Para kernels simétricos os resultados coincidem; para Sobel $G_x$ (assimétrico) a diferença é nítida.

#### 📤 Saída

Para uma dada matriz de entrada e kernel, imprima a matriz resultante da correlação e da convolução.


In [7]:
#| label: fig-ep03_06-sim
#| fig-cap: "Simulador: Correlação vs Convolução (Sobel Gx)"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_06_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><strong>Imagem 5x5</strong><div id="ep03_06_img"></div><button id="ep03_06_rand">Random 5x5</button></div>
    <div style="display: grid; grid-template-columns: 1fr 1fr;">
        <div><strong>Correlação com Gx</strong><div id="ep03_06_corr"></div></div>
        <div><strong>Convolução com Gx</strong><div id="ep03_06_conv"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    const Gx = [[-1,0,1],[-2,0,2],[-1,0,1]];
    let img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256));
    function applyKernel(img, kernel, isCorr) {
        let k = kernel.map(row=>[...row]);
        if(!isCorr) k = k.map(row=>row.reverse()).reverse();
        let res = Array(25).fill(0);
        for(let i=1;i<=3;i++) for(let j=1;j<=3;j++) {
            let sum=0;
            for(let di=-1;di<=1;di++) for(let dj=-1;dj<=1;dj++)
                sum += k[di+1][dj+1] * img[(i+di)*5 + (j+dj)];
            res[i*5+j] = Math.min(255, Math.max(0, Math.round(sum)));
        }
        return res;
    }
    function render() {
        let corr = applyKernel(img, Gx, true);
        let conv = applyKernel(img, Gx, false);
        function draw(container, arr) {
            container.innerHTML='';
            for(let i=0;i<25;i++) {
                let v = arr[i];
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            }
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(5,42px)';
        }
        draw(document.getElementById('ep03_06_img'), img);
        draw(document.getElementById('ep03_06_corr'), corr);
        draw(document.getElementById('ep03_06_conv'), conv);
    }
    document.getElementById('ep03_06_rand').onclick = () => { img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256)); render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_06.py
# Código Python

In [ ]:
TestSuite("EP03_06.py").run()

### EP03_07 🌫️ Filtro de Média vs Gaussiano

Implemente os filtros de média (kernel uniforme) e Gaussiano (utilizando a fórmula discreta) e compare o MSE entre eles e a imagem original.

#### 📌 Parâmetros

- Filtro média: kernel $n\times n$ com coeficientes $1/n^2$.
- Filtro Gaussiano: $G(s,t) = \frac{1}{2\pi\sigma^2}e^{-(s^2+t^2)/(2\sigma^2)}$, normalizado soma 1.

#### 📥 Entrada

$L$, $C$, matriz, $n$ (ímpar), $\sigma$. Saída: matriz média, matriz Gaussiana.


In [8]:
#| label: fig-ep03_07-sim
#| fig-cap: "Simulador: Média vs Gaussiano"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_07_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_07_rand">Gerar Imagem</button> Kernel size: <input id="ep03_07_ksize" value="3" style="width:40px;"> σ: <input id="ep03_07_sigma" value="1.0" style="width:40px;"></div>
    <div style="display: grid; grid-template-columns: repeat(3,1fr);">
        <div><strong>Original</strong><div id="ep03_07_orig"></div></div>
        <div><strong>Média</strong><div id="ep03_07_mean"></div></div>
        <div><strong>Gaussiano</strong><div id="ep03_07_gauss"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256));
    function gaussKernel(size, sigma) {
        let k = Array(size).fill().map(()=>Array(size).fill(0));
        let sum=0, mid=Math.floor(size/2);
        for(let i=0;i<size;i++) for(let j=0;j<size;j++) {
            let dx=i-mid, dy=j-mid;
            let v = Math.exp(-(dx*dx+dy*dy)/(2*sigma*sigma));
            k[i][j]=v; sum+=v;
        }
        for(let i=0;i<size;i++) for(let j=0;j<size;j++) k[i][j]/=sum;
        return k;
    }
    function applyFilter(img, kernel) {
        let size=kernel.length, off=Math.floor(size/2), res=Array(25).fill(0);
        for(let i=off;i<5-off;i++) for(let j=off;j<5-off;j++) {
            let sum=0;
            for(let di=-off;di<=off;di++) for(let dj=-off;dj<=off;dj++)
                sum += kernel[di+off][dj+off] * img[(i+di)*5+(j+dj)];
            res[i*5+j]=Math.min(255,Math.max(0,Math.round(sum)));
        }
        return res;
    }
    function render() {
        let ksize = parseInt(document.getElementById('ep03_07_ksize').value) || 3;
        let sigma = parseFloat(document.getElementById('ep03_07_sigma').value) || 1;
        let meanKernel = Array(ksize).fill().map(()=>Array(ksize).fill(1/(ksize*ksize)));
        let gaussK = gaussKernel(ksize, sigma);
        let meanImg = applyFilter(img, meanKernel);
        let gaussImg = applyFilter(img, gaussK);
        function draw(container, arr) {
            container.innerHTML='';
            for(let i=0;i<25;i++) {
                let v=arr[i];
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            }
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(5,42px)';
        }
        draw(document.getElementById('ep03_07_orig'), img);
        draw(document.getElementById('ep03_07_mean'), meanImg);
        draw(document.getElementById('ep03_07_gauss'), gaussImg);
    }
    document.getElementById('ep03_07_rand').onclick = () => { img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256)); render(); };
    document.getElementById('ep03_07_ksize').oninput = render;
    document.getElementById('ep03_07_sigma').oninput = render;
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_07.py
# Código Python

In [ ]:
TestSuite("EP03_07.py").run()

### EP03_08 ⚡ Realce Laplaciano (w4 e w8)

Implemente o realce pela subtração do Laplaciano: $g = f - \nabla^2 f$, utilizando os kernels $w_4$ e $w_8$.

#### Kernels

$$ w_4 = \begin{bmatrix}0 & 1 & 0\\1 & -4 & 1\\0 & 1 & 0\end{bmatrix}, \quad w_8 = \begin{bmatrix}1 & 1 & 1\\1 & -8 & 1\\1 & 1 & 1\end{bmatrix} $$

#### 📤 Saída

Duas matrizes: realce com $w_4$ e realce com $w_8$ (valores saturarados em [0,255]).


In [9]:
#| label: fig-ep03_08-sim
#| fig-cap: "Simulador: Realce Laplaciano"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_08_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_08_rand">Nova Imagem 5x5</button></div>
    <div style="display: grid; grid-template-columns: repeat(3,1fr);">
        <div><strong>Original</strong><div id="ep03_08_orig"></div></div>
        <div><strong>Realce w4</strong><div id="ep03_08_w4"></div></div>
        <div><strong>Realce w8</strong><div id="ep03_08_w8"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256));
    const w4 = [[0,1,0],[1,-4,1],[0,1,0]];
    const w8 = [[1,1,1],[1,-8,1],[1,1,1]];
    function lap(img, kernel) {
        let res = Array(25).fill(0);
        for(let i=1;i<=3;i++) for(let j=1;j<=3;j++) {
            let lapVal=0;
            for(let di=-1;di<=1;di++) for(let dj=-1;dj<=1;dj++)
                lapVal += kernel[di+1][dj+1] * img[(i+di)*5+(j+dj)];
            let sharp = img[i*5+j] - lapVal;
            res[i*5+j] = Math.min(255, Math.max(0, Math.round(sharp)));
        }
        return res;
    }
    function render() {
        let w4res = lap(img, w4);
        let w8res = lap(img, w8);
        function draw(container, arr) {
            container.innerHTML='';
            for(let i=0;i<25;i++) {
                let v=arr[i];
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            }
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(5,42px)';
        }
        draw(document.getElementById('ep03_08_orig'), img);
        draw(document.getElementById('ep03_08_w4'), w4res);
        draw(document.getElementById('ep03_08_w8'), w8res);
    }
    document.getElementById('ep03_08_rand').onclick = () => { img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256)); render(); };
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_08.py
# Código Python

In [ ]:
TestSuite("EP03_08.py").run()

### EP03_09 🔪 Unsharp Masking (USM)

Implemente o realce por *unsharp masking*: $g = f + k \cdot (f - f_{\text{gauss}})$. O parâmetro $k$ controla a intensidade.

#### 📐 Fórmula

$$ g = f + k \cdot (f - G_\sigma * f) $$

#### 📥 Parâmetros

$L$, $C$, imagem, $\sigma$ (desvio Gaussiano), $k$ (fator de realce). Saída: imagem realçada.


In [10]:
#| label: fig-ep03_09-sim
#| fig-cap: "Simulador: Unsharp Masking"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_09_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_09_rand">Nova Imagem</button> σ: <input id="ep03_09_sigma" value="1.0" style="width:50px;"> k: <input id="ep03_09_k" value="1.0" style="width:50px;"></div>
    <div style="display: grid; grid-template-columns: 1fr 1fr;">
        <div><strong>Original</strong><div id="ep03_09_orig"></div></div>
        <div><strong>USM</strong><div id="ep03_09_usm"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256));
    function gaussFilter(img, sigma) {
        let size=3, off=1, kernel=[], sum=0;
        for(let i=-1;i<=1;i++) for(let j=-1;j<=1;j++) {
            let v = Math.exp(-(i*i+j*j)/(2*sigma*sigma));
            kernel.push(v); sum+=v;
        }
        kernel = kernel.map(v=>v/sum);
        let res = Array(25).fill(0);
        for(let i=1;i<=3;i++) for(let j=1;j<=3;j++) {
            let val=0, idx=0;
            for(let di=-1;di<=1;di++) for(let dj=-1;dj<=1;dj++)
                val += kernel[idx++] * img[(i+di)*5+(j+dj)];
            res[i*5+j]=val;
        }
        return res;
    }
    function render() {
        let sigma = parseFloat(document.getElementById('ep03_09_sigma').value) || 1;
        let k = parseFloat(document.getElementById('ep03_09_k').value) || 1;
        let blurred = gaussFilter(img, sigma);
        let usm = img.map((v,i) => {
            let val = v + k * (v - blurred[i]);
            return Math.min(255, Math.max(0, Math.round(val)));
        });
        function draw(container, arr) {
            container.innerHTML='';
            for(let i=0;i<25;i++) {
                let v=arr[i];
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            }
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(5,42px)';
        }
        draw(document.getElementById('ep03_09_orig'), img);
        draw(document.getElementById('ep03_09_usm'), usm);
    }
    document.getElementById('ep03_09_rand').onclick = () => { img = Array(25).fill(0).map(()=>Math.floor(Math.random()*256)); render(); };
    document.getElementById('ep03_09_sigma').oninput = render;
    document.getElementById('ep03_09_k').oninput = render;
    render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_09.py
# Código Python

In [ ]:
TestSuite("EP03_09.py").run()

### EP03_10 🧂 Filtro da Mediana para Ruído Sal e Pimenta

Implemente o filtro da mediana (janela $n\times n$) e compare com o filtro da média na remoção de ruído impulsivo. A entrada conterá uma imagem corrompida com ruído sal e pimenta (densidade conhecida).

#### 🧮 Mediana

$$ g(x,y) = \text{median}\{ f(x+s, y+t) : -a \le s,t \le a \} $$

#### 📤 Saída

Duas matrizes: imagem filtrada pela mediana e imagem filtrada pela média (kernel $n\times n$).


In [11]:
#| label: fig-ep03_10-sim
#| fig-cap: "Simulador: Mediana vs Média para Ruído Sal/Pimenta"
#| echo: false

from IPython.display import HTML
HTML("""
<div id="ep03_10_sim" style="background: #fef9ef; border-radius: 18px; padding: 10px;">
    <div><button id="ep03_10_rand">Gerar Imagem + Ruído 10%</button> Janela: <input id="ep03_10_size" value="3" style="width:40px;"></div>
    <div style="display: grid; grid-template-columns: repeat(3,1fr);">
        <div><strong>Original</strong><div id="ep03_10_orig"></div></div>
        <div><strong>Ruído + Pimenta</strong><div id="ep03_10_noisy"></div></div>
        <div><strong>Mediana</strong><div id="ep03_10_median"></div></div>
        <div><strong>Média</strong><div id="ep03_10_mean"></div></div>
    </div>
</div>
<script>
setTimeout(() => {
    let clean = Array(25).fill(0).map(()=>Math.floor(Math.random()*256));
    let noisy = [...clean];
    function addNoise() {
        noisy = clean.map(v=> Math.random()<0.1 ? (Math.random()<0.5?0:255) : v);
    }
    function medianFilter(arr, size) {
        let off=Math.floor(size/2), res=Array(25).fill(0);
        for(let i=off;i<5-off;i++) for(let j=off;j<5-off;j++) {
            let vals=[];
            for(let di=-off;di<=off;di++) for(let dj=-off;dj<=off;dj++)
                vals.push(arr[(i+di)*5+(j+dj)]);
            vals.sort((a,b)=>a-b);
            res[i*5+j]=vals[Math.floor(vals.length/2)];
        }
        return res;
    }
    function meanFilter(arr, size) {
        let off=Math.floor(size/2), res=Array(25).fill(0);
        for(let i=off;i<5-off;i++) for(let j=off;j<5-off;j++) {
            let sum=0;
            for(let di=-off;di<=off;di++) for(let dj=-off;dj<=off;dj++)
                sum += arr[(i+di)*5+(j+dj)];
            res[i*5+j]=Math.min(255,Math.max(0,Math.round(sum/(size*size))));
        }
        return res;
    }
    function render() {
        let size = parseInt(document.getElementById('ep03_10_size').value) || 3;
        let med = medianFilter(noisy, size);
        let mean = meanFilter(noisy, size);
        function draw(container, arr) {
            container.innerHTML='';
            for(let i=0;i<25;i++) {
                let v=arr[i];
                let d=document.createElement('div');
                d.style.cssText='width:42px;height:42px;display:flex;align-items:center;justify-content:center;background:rgb('+v+','+v+','+v+');color:'+(v>128?'black':'white')+';border:1px solid #ccc;';
                d.innerText=v;
                container.appendChild(d);
            }
            container.style.display='grid'; container.style.gridTemplateColumns='repeat(5,42px)';
        }
        draw(document.getElementById('ep03_10_orig'), clean);
        draw(document.getElementById('ep03_10_noisy'), noisy);
        draw(document.getElementById('ep03_10_median'), med);
        draw(document.getElementById('ep03_10_mean'), mean);
    }
    document.getElementById('ep03_10_rand').onclick = () => { clean = Array(25).fill(0).map(()=>Math.floor(Math.random()*256)); addNoise(); render(); };
    document.getElementById('ep03_10_size').oninput = render;
    addNoise(); render();
}, 100);
</script>
""")

In [ ]:
%%writefile EP03_10.py
# Código Python

In [ ]:
TestSuite("EP03_10.py").run()